# Chapter 1 · Notebook 3 of 3 — Observability & Cost

**AI Agent Course** · Nebius Token Factory × NVIDIA

In Notebook 2 we built our first agentic loop. This notebook asks the engineering question: **is that loop actually usable?** Two checks decide it: 
1. Is it *fast enough* (latency)
2. Is it *cheap enough* (cost). 

By the end you can measure both for any call you make.

*(Runs standalone — the setup below is the same as the previous notebooks'. If you're continuing in the same session, skip to Section 1.)*

## Setup

1. Go to [tokenfactory.nebius.com](https://tokenfactory.nebius.com) and create an account.
2. Open **Get API Key → Create API key** and copy it (you can't view it again later).

<details>
<summary>Show screenshot: Token Factory home</summary>

![Token Factory home](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/token-factory-home.png)

</details>

<details>
<summary>Show screenshot: API Key creation</summary>

![API Key creation](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/api-key-creation.png)

</details>

Your `lesson/.env` should look like:

```
NEBIUS_API_KEY=paste-your-key-here
```

**Why a `.env` file at all?** In real development, configuration that changes between people and machines (and *especially* secrets) lives outside your code. The `.env` file holds it; `load_dotenv()` reads it into environment variables. That way the same code runs on your laptop, your teammate's, and the server, and no key ever gets committed. **Key hygiene reminder:** keys live in `.env`, never in code, and `.env` goes in `.gitignore`. Never commit a notebook with a real key in it, unless you want someone stealing your Token Factory credits!

**Running in Google Colab?** Colab doesn't have your `.env`. Two options:

- Upload it: open the file browser (folder icon, left side) and drag your `.env` in, then point `load_dotenv()` at it with load_dotenv("path-to-your-env-file").
- Or use **Colab Secrets** (key icon, left side): add `NEBIUS_API_KEY` there, then run `from google.colab import userdata; os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")` instead of `load_dotenv()`.

In [ ]:
%pip install -q openai python-dotenv rich sympy

In [ ]:
import os

from dotenv import load_dotenv
from rich import print

load_dotenv("lesson/.env")
%load_ext rich

# Colab users: comment the two lines above and use Colab Secrets instead:
# from google.colab import userdata
# os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")

assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in lesson/.env"
print("Keys loaded.")

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.us-central1.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

## 1. Observability: is the loop fast enough?

Over time, we will use models, evaluate them, change models, and repeat. To evaluate and compare these models, we need to understand their performance. Token Factory has built-in observability to make this easy.

**The three latency numbers**:

- **TTFT:** *Time To First Token*. How 'snappy' and responsive the model feels. **When it matters:** a customer-facing chatbot, for example. If nothing appears for three seconds, the user assumes it's broken and leaves.
- **t/s:** *Tokens per Second*. How fast does the model feel while it is giving a response? **When it matters:** long generations streaming to a reader. A report being written on screen at 10 t/s feels slow and frustrating; at 60 t/s it feels alive.
- **E2EL:** *End-to-End Latency*. How long from request to finished output? **When it matters:** whenever nobody is watching the stream! In our agentic loops, we are not watching out model at all times. But it does not mean that the total response time does not matter - if our agent is slow, it won't be as effective.

These numbers can be viewed under any response in the Playground:

<details>
<summary>Show screenshot: Playground metrics</summary>

![Playground metrics](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/request-metrics.png)

</details>

Chat Completions responses don't include these metrics, latency is something you measure from the client side. A small wrapper that does it for you (streaming + a stopwatch) is in the **Appendix (Optional)** at the end of this notebook.

### Exercise: For the three agents below, determine which latency metric would be the most important and why.

1. **A live customer-support assistant.** A user types a question into a chat widget on a website and watches the reply stream in. Most answers are 2–3 sentences.

2. **A nightly code-review agent.** It wakes up at 2am, pulls every PR opened that day, and for each one runs a multi-step loop: read the diff, look up related files, draft comments. Nobody sees the output until they open their laptop in the morning, but it has to finish before a 9am standup.

3. **A research assistant that drafts long-form reports.** The user gives it a topic, and it writes a 2,000-word summary directly into the browser. The user sits and reads along as the text appears.

Answer:

## 2. Costs: is the loop cheap enough?

**LLMs can get expensive!** Every time the model is receiving input and producing output, it is actively billing. This is why cost management becomes **extremely important as our model scales**. Later, we will discuss best practices, so our agent does not break the bank.

The math itself is simple - the card gives you two prices, the response gives you two token counts:

$$\text{cost} = \frac{\text{prompt tokens} \times \text{price}_{in} + \text{completion tokens} \times \text{price}_{out}}{1{,}000{,}000}$$

A prototype request that costs a fraction of a cent feels free. The same request at 100,000 calls/day is a lot of money! When agents are running wild, we have less control over how much they use. Therefore, it is a great habit to **know what every request costs.** Let's price one real request:

In [ ]:
# Prices from the Nemotron 3 Nano model card, $/1M tokens
PRICE_IN = 0.06
PRICE_OUT = 0.24

resp = client.chat.completions.create(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[{"role": "user", "content": "Summarize the outlook for the specialty coffee market in one paragraph."}],
)

u = resp.usage
cost = (u.prompt_tokens * PRICE_IN + u.completion_tokens * PRICE_OUT) / 1e6

print(f"input tokens:  {u.prompt_tokens}")
print(f"output tokens: {u.completion_tokens}")
print(f"cost:          ${cost:.6f}")

### Exercise: How is our bank feeling?

Calculate the cost per request for 2 models using the request above ( `messages=[{"role": "user", "content": "Summarize the outlook for the specialty coffee market in one paragraph."}]` ):

1. `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B` (prices above)
2. `nvidia/Nemotron-3-Ultra-550b-a55b` (you will need to find its pricing on the card!)

What is the average cost per request on each? Per 1,000 requests? Per 100,000? At what scale does the gap stop feeling academic?

Answer:

### Watching spend in the Token Factory console

You don't have to track everything yourself, Token Factory records every billable token. To pull up your own spend:

1. In the **left-hand panel**, open **Billing settings → Usage**.

<details>
<summary>Show screenshot: Usage button</summary>

![Usage button](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/usage-button.png)

</details>

2. Switch the view to **Chart**.
3. Set **Group by → Products**.
4. Open the **Products** dropdown and select **`NVIDIA-Nemotron-3-Nano-30B-A3B Input`** and **`NVIDIA-Nemotron-3-Nano-30B-A3B Output`**.

<details>
<summary>Show screenshot: Billing dropdown</summary>

![Billing dropdown](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/billing-dropdown.png)

</details>

Each day's bar now splits into the two products you selected:

<details>
<summary>Show screenshot: Daily spend, input vs output</summary>

![Daily spend, input vs output](https://raw.githubusercontent.com/Nebius-Academy/agentic-ai-course-assets/main/chapter-01-token-factory/billing-graph.png)

</details>

Two things to notice in this chart:

- **Input and output are separate products.**  This is because they're billed at separate rates, exactly as the model card said. Output tokens cost 4× more on Nano, so even modest generations dominate costs. Capping `max_tokens` pays off right here.
- **The total up top is the ground truth.** Your own per-request estimates should agree with this page.

If you've been running these notebooks, today's bar should include every call you've made this chapter. Two habits from day one: check this chart **per key** (each project gets its own key, so spend is attributable), and **set a spend limit per key** so a runaway loop can't burn the budget - this will be extremely important in future chapters.

## 3. Wrap-up

You started this chapter with an account and nothing else. Across three notebooks, here's what you built:

- **Chose a model like an engineer.**  Read the card, compared Nano to its bigger siblings, and picked based on the workload, not the hype.
- **Called it from Python** through the OpenAI-compatible API: single calls, system-prompt voices, and multi-turn conversations where you manage the history.
- **Built your first agentic loop.** The model requested an action, your code validated and executed it, and the result went back. You also broke it to see how fragile it can be.
- **Put numbers on the loop.** TTFT, t/s, and E2EL for speed; token counts and two prices for cost; ground truth in the billing console.

If one idea should stick, it's this: **treat every model call as a priced, measurable transaction.** Tokens in, tokens out, with a dollar figure and a latency number on every one. Agents are going to make *thousands* of these calls on your behalf! The habits from this chapter (log everything, know your costs) are what keep that from becoming a very expensive surprise.

**Where we're headed:** our calculator worked, but our weather tool was fake. The model asked for data nobody could fetch. Next chapter we plug in the first *real* tool: **web search via Tavily**. Combined with the loop you built in Notebook 2, that gives us a genuine agent — a model that can decide it doesn't know something, go find out, and come back with an answer.

*See you in Chapter 2!*

---

## Appendix (Optional): measuring and tracking from code

*Everything below is extra depth for those who want it! The main learning path is complete without it.*

### A.1 A latency wrapper for Chat Completions


Chat Completions doesn't report TTFT/t/s, but with streaming and a stopwatch, you can measure them yourself. This is genuinely how production clients do it:

In [ ]:
import time

def measure(model: str, prompt: str):
    """Measure TTFT, t/s, and E2EL for one streaming request."""
    t_start = time.perf_counter()
    t_first = None
    chunks = []

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        stream_options={"include_usage": True},
    )

    usage = None
    for chunk in stream:
        if chunk.usage:
            usage = chunk.usage
            continue
        delta = chunk.choices[0].delta.content
        if delta:
            if t_first is None:
                t_first = time.perf_counter()   # <-- the moment the first token lands
            chunks.append(delta)

    t_end = time.perf_counter()

    ttft = t_first - t_start
    e2el = t_end - t_start
    out_tokens = usage.completion_tokens if usage else len(chunks)
    tps = out_tokens / (t_end - t_first)

    print(f"model:  {model}")
    print(f"TTFT:   {ttft:.2f}s")
    print(f"t/s:    {tps:.1f}")
    print(f"E2EL:   {e2el:.2f}s   ({out_tokens} output tokens)")
    return "".join(chunks)

_ = measure(
    "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    "Explain what an AI agent is in one paragraph.",
)

### A.2 A cost wrapper for every request

The per-request calculation from Section 2, packaged so every call reports its own price:

In [ ]:
def tracked_completion(**kwargs):
    """Run a chat completion and print what it cost."""
    resp = client.chat.completions.create(**kwargs)
    u = resp.usage
    cost = (u.prompt_tokens * PRICE_IN + u.completion_tokens * PRICE_OUT) / 1e6
    print(f"input tokens:  {u.prompt_tokens}")
    print(f"output tokens: {u.completion_tokens}")
    print(f"cost:          ${cost:.6f}")
    return resp, cost

resp, cost = tracked_completion(
    model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    messages=[{"role": "user", "content": "Summarize the outlook for the specialty coffee market in one paragraph."}],
)

### A.3 One request seems cheap, but what about when we scale?

In [ ]:
# 1 a day? 1,000? 100,000? 1,000,000?:
for requests_per_day in [1, 1_000, 100_000, 1_000_000]:
    print(f"{requests_per_day:>9,} requests/day  →  ${cost * requests_per_day:>10,.2f}/day  →  ${cost * requests_per_day * 30:>12,.2f}/month")